# Voice Activity Detection

Build a call, add noise, and watch the state machine turn probabilities into START and END events. Predict before you run.

In [ ]:
import sys
sys.path.insert(0, '../code')
import numpy as np
import main

dialog = main.synthetic_dialog()
noisy = main.mix_at_snr(dialog['signal'], main.noise('babble', len(dialog['signal'])), 10, dialog['speech'])
probs = main.speech_probability(noisy)
result = main.run_vad(probs)
print(len(dialog['utterances']), 'utterances ->', len(result.segments), 'segments')
result.events[:6]

## Noise type and SNR

Which noise type produces the most false alarms, and does it improve as the SNR rises?

In [ ]:
for kind in ('white', 'hum', 'babble'):
    for snr in (20, 10, 5, 2.5):
        x = main.mix_at_snr(dialog['signal'], main.noise(kind, len(dialog['signal'])), snr, dialog['speech'])
        m = main.evaluate(x, dialog)
        print(f"{kind:>6} {snr:>4} dB  F1 {m['f1']:.2f}  false alarms {100 * m['false_alarm']:4.1f}%")

## The timers

`min_silence_duration` is both the hangover that keeps a chunked phone number together and the delay before END_OF_SPEECH. Find the smallest value that still gives one segment per utterance.

In [ ]:
clean = main.mix_at_snr(dialog['signal'], main.noise('white', len(dialog['signal'])), 20, dialog['speech'])
for silence in (0.1, 0.25, 0.4, 0.55, 0.8):
    m = main.evaluate(clean, dialog, main.VADParams(min_silence_duration=silence))
    print(f"min_silence_duration={silence:<4} segments={m['segments']:>2}  END after {1000 * m['end_delay_s']:.0f} ms")

## Prefix padding

Make START slower with `min_speech_duration=0.15`, then give the onset back with padding.

In [ ]:
for pad in (0.0, 0.1, 0.2, 0.5):
    m = main.evaluate(clean, dialog, main.VADParams(min_speech_duration=0.15, prefix_padding_duration=pad))
    print(f"prefix_padding_duration={pad:<4} onset lost {m['onset_clipping_ms']:5.1f} ms")